# 01 — Quickstart

This notebook shows the minimal usage of `internals_extraction`:
- Import the plugin (zero model changes)
- Run inference as normal
- Inspect all extracted signals

**Requirements:** `pip install transformers torch internals-extraction`

In [ ]:
import internals_extraction          # ← the only line needed
from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np

print("Plugin active:", internals_extraction.config)

## 1. Load a model (unchanged)

In [ ]:
MODEL = "gpt2"   # swap for any causal-LM

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model     = AutoModelForCausalLM.from_pretrained(MODEL)
model.eval()
print(f"Loaded {MODEL} — {sum(p.numel() for p in model.parameters()):,} parameters")

## 2. Run inference as normal

In [ ]:
PROMPT = "The Eiffel Tower is located in the city of"

inputs     = tokenizer(PROMPT, return_tensors="pt")
output_ids = model.generate(**inputs, max_new_tokens=10, do_sample=False)

print("Generated:", tokenizer.decode(output_ids[0], skip_special_tokens=True))

## 3. Retrieve the InternalsRun

In [ ]:
run = internals_extraction.get_latest()
print(run)

## 4. Hidden states

Layer 0 is the embedding output; layers 1..N are transformer block outputs.

In [ ]:
# Per-layer mean hidden state of prompt tokens — (num_layers, batch, hidden)
in_mean = run.input_hidden_states_mean
print(f"input_hidden_states_mean  shape: {in_mean.shape}")

# Per-layer mean hidden state of generated tokens
out_mean = run.output_hidden_states_mean
print(f"output_hidden_states_mean shape: {out_mean.shape}")

# Full hidden states per layer for the prompt
print(f"input_hidden_states[6]    shape: {run.input_hidden_states[6].shape}")
print(f"  (batch, prompt_len, hidden)")

In [ ]:
# Cosine similarity between consecutive layers (prompt mean representations)
means = run.input_hidden_states_mean[:, 0, :]   # (L, hidden)

print("Layer-to-layer cosine similarity (input token mean):")
for i in range(len(means) - 1):
    a, b = means[i], means[i + 1]
    cos  = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)
    bar  = "█" * int(cos * 30)
    print(f"  Layer {i:2d} → {i+1:2d}  cos={cos:.4f}  {bar}")

## 5. Attentions

`run.attentions[step][layer]` → `(batch, seq_q, seq_k)` (head-averaged by default).

In [ ]:
input_tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
input_len    = run.input_len

print(f"Number of generation steps captured: {len(run.attentions)}")
print(f"Number of layers per step: {len(run.attentions[0])}")
print(f"Attention shape at step 0, layer 0: {run.attentions[0][0].shape}")
print()

# Where does the last prompt token attend? (step 0, each layer)
print(f"Top-2 attention targets for token {input_tokens[-1]!r} per layer:")
for layer_idx, attn in enumerate(run.attentions[0]):
    row     = attn[0, -1, :]                          # (seq_k,)
    top2    = row.argsort()[-2:][::-1]
    targets = [f"{input_tokens[j]!r}({row[j]:.2f})" for j in top2 if j < input_len]
    print(f"  Layer {layer_idx:2d}: {', '.join(targets)}")

## 6. Logits

`run.logits` → `(num_steps, batch, vocab_size)` — last-position logits at each generation step.

In [ ]:
def softmax(x):
    x = x - x.max()
    e = np.exp(x)
    return e / e.sum()

output_tokens = tokenizer.convert_ids_to_tokens(output_ids[0, input_len:])

print(f"Logits shape: {run.logits.shape}")
print()
print(f"{'Step':>4}  {'Generated':>15}  {'Top prediction':>15}  {'Prob':>6}")
print(f"{'────':>4}  {'─────────':>15}  {'──────────────':>15}  {'────':>6}")
for step, logit_row in enumerate(run.logits):
    probs    = softmax(logit_row[0])
    top_id   = probs.argmax()
    top_tok  = tokenizer.decode([top_id])
    gen_tok  = output_tokens[step] if step < len(output_tokens) else "—"
    print(f"{step:>4}  {gen_tok!r:>15}  {top_tok!r:>15}  {probs[top_id]:>6.3f}")

## 7. Logit-lens (optional)

Apply the LM-head to every layer's hidden state to see what the model "would have predicted" at each depth. Computed on CPU in the background — no extra GPU memory.

In [ ]:
# Enable and run again
internals_extraction.register_model(model)
internals_extraction.config.extract_logit_lens = True

inputs2 = tokenizer("The capital of France is", return_tensors="pt")
model.generate(**inputs2, max_new_tokens=1, do_sample=False)

run2 = internals_extraction.get_latest()

print("Logit-lens — top prediction at the last prompt position per layer:")
print(f"  {'Layer':>6}  Top-3 predictions")
print(f"  {'─────':>6}  ─────────────────")
for i, lens in enumerate(run2.logit_lens):
    probs    = softmax(lens[0, -1, :])
    top3_ids = probs.argsort()[-3:][::-1]
    top3     = [f"{tokenizer.decode([t])!r}({probs[t]:.3f})" for t in top3_ids]
    marker   = "  ◀" if any("Paris" in tokenizer.decode([t]) for t in top3_ids) else ""
    print(f"  {i:>6}  {', '.join(top3)}{marker}")

## 8. Accessing multiple runs

In [ ]:
for prompt in ["Dogs are", "The moon is made of", "Python is a"]:
    inp = tokenizer(prompt, return_tensors="pt")
    model.generate(**inp, max_new_tokens=5, do_sample=False)

all_runs = internals_extraction.get_all()
print(f"Runs in buffer: {len(all_runs)}")
for r in all_runs:
    print(f"  {r}")